# Sistemas Inteligentes I
## Búsqueda adversarial: poda Alfa–Beta

**Autor:** Jairo I. Vélez B.  
**Desarrollado y resuelto por:** Juan David Ocampo González

---


# 1. Punto de partida: el problema de Minimax

Minimax supone que:

- MAX intenta maximizar;
- MIN intenta minimizar;
- ambos jugadores actúan racionalmente.

El problema es que, para garantizar su decisión, Minimax puede explorar aproximadamente:

$$O(b^m)$$

nodos.

Sin embargo, algunas ramas pueden resultar irrelevantes.

La idea central de Alfa–Beta es:

> **dejar de explorar una rama cuando ya sabemos que no puede mejorar la decisión de un jugador.**

La poda no cambia el resultado de Minimax.  
Solo intenta obtenerlo explorando menos nodos.


# 2. Recordatorio: un árbol de juego

Utilizaremos inicialmente el mismo tipo de representación:

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Minimax calcula:

- `B = 2`
- `C = 1`
- `D = 6`

y finalmente:

$$A=\max(2,1,6)=6$$


In [1]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades


({'A': ['B', 'C', 'D'],
  'B': ['B1', 'B2', 'B3'],
  'C': ['C1', 'C2', 'C3'],
  'D': ['D1', 'D2', 'D3']},
 {'B1': 3,
  'B2': 5,
  'B3': 2,
  'C1': 9,
  'C2': 1,
  'C3': 4,
  'D1': 6,
  'D2': 7,
  'D3': 8})

# 3. ¿Qué representan alfa y beta?

Durante la búsqueda mantenemos dos límites.

### Alfa — $\alpha$

Es el mejor valor que **MAX puede garantizar hasta el momento**.

Inicialmente:

$$\alpha=-\infty$$

### Beta — $\beta$

Es el mejor valor que **MIN puede garantizar hasta el momento**.

Inicialmente:

$$\beta=+\infty$$

Durante la búsqueda:

- MAX actualiza $\alpha$;
- MIN actualiza $\beta$.

Cuando ocurre:

$$\boxed{\alpha \geq \beta}$$

podemos realizar una **poda**.


# 4. Intuición de una poda

Suponga que MAX ya dispone de una alternativa con valor `6`.

Ahora explora otra rama cuyo turno pertenece a MIN.

Si MIN encuentra dentro de esa rama una opción con valor `4`, sabemos que podrá forzar:

$$valor \leq 4$$

MAX ya dispone de `6`, por lo que nunca elegirá una alternativa que termine en
`4` o menos.

Por tanto:

> **el resto de esa rama ya no puede cambiar la decisión de MAX.**

Podemos dejar de explorarla.


# 5. Implementación de Alfa–Beta

La estructura es muy similar a Minimax.

La diferencia está en que conservamos y actualizamos los límites
$\alpha$ y $\beta$.


In [2]:
import time
from math import inf

def alfa_beta(nodo, es_max, arbol, utilidades, alfa=-inf, beta=inf):
    if nodo in utilidades:
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta(hijo, False, arbol, utilidades, alfa, beta)
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta(hijo, True, arbol, utilidades, alfa, beta)
            )

            beta = min(beta, valor)

            if alfa >= beta:
                break

        return valor


alfa_beta("A", True, arbol, utilidades)


6

# 6. Comparar Alfa–Beta con Minimax

Primero implementaremos una versión sencilla de Minimax.


In [3]:
def minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo]

    valores = [
        minimax(hijo, not es_max, arbol, utilidades)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


print("Minimax   :", minimax("A", True, arbol, utilidades))
print("Alfa-Beta :", alfa_beta("A", True, arbol, utilidades))


Minimax   : 6
Alfa-Beta : 6


# 7. Alfa–Beta paso a paso

Ahora imprimiremos:

- nodo visitado;
- jugador;
- valor de $\alpha$;
- valor de $\beta$;
- momento en el que se produce una poda.


In [4]:
def alfa_beta_debug(
    nodo,
    es_max,
    arbol,
    utilidades,
    alfa=-inf,
    beta=inf,
    profundidad=0
):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(
            f"{sangria}{nodo}: terminal = {utilidades[nodo]} "
            f"[α={alfa}, β={beta}]"
        )
        return utilidades[nodo]

    print(
        f"{sangria}{nodo}: {jugador} "
        f"[α={alfa}, β={beta}]"
    )

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                False,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = max(valor, valor_hijo)
            alfa = max(alfa, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                True,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = min(valor, valor_hijo)
            beta = min(beta, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor


alfa_beta_debug("A", True, arbol, utilidades)


A: MAX [α=-inf, β=inf]
    B: MIN [α=-inf, β=inf]
        B1: terminal = 3 [α=-inf, β=inf]
      después de B1: valor=3, α=-inf, β=3
        B2: terminal = 5 [α=-inf, β=3]
      después de B2: valor=3, α=-inf, β=3
        B3: terminal = 2 [α=-inf, β=3]
      después de B3: valor=2, α=-inf, β=2
  después de B: valor=2, α=2, β=inf
    C: MIN [α=2, β=inf]
        C1: terminal = 9 [α=2, β=inf]
      después de C1: valor=9, α=2, β=9
        C2: terminal = 1 [α=2, β=9]
      después de C2: valor=1, α=2, β=1
      PODA en C: α=2 >= β=1
  después de C: valor=2, α=2, β=inf
    D: MIN [α=2, β=inf]
        D1: terminal = 6 [α=2, β=inf]
      después de D1: valor=6, α=2, β=6
        D2: terminal = 7 [α=2, β=6]
      después de D2: valor=6, α=2, β=6
        D3: terminal = 8 [α=2, β=6]
      después de D3: valor=6, α=2, β=6
  después de D: valor=6, α=6, β=inf


6

# 8. Un ejemplo diseñado para observar podas

El orden de los valores del árbol anterior no siempre produce una poda muy visible.

Usaremos ahora este árbol:

```text
                         A (MAX)
                    /             \
               B (MIN)           C (MIN)
              /      \           /      \
          D(MAX)   E(MAX)    F(MAX)    G(MAX)
           3  5     6  9      1  2      0 -1
```

La exploración se realiza de izquierda a derecha.


In [5]:
arbol_poda = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_poda = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

print("Minimax:", minimax("A", True, arbol_poda, utilidades_poda))
print()
alfa_beta_debug("A", True, arbol_poda, utilidades_poda)


Minimax: 5

A: MAX [α=-inf, β=inf]
    B: MIN [α=-inf, β=inf]
        D: MAX [α=-inf, β=inf]
            D1: terminal = 3 [α=-inf, β=inf]
          después de D1: valor=3, α=3, β=inf
            D2: terminal = 5 [α=3, β=inf]
          después de D2: valor=5, α=5, β=inf
      después de D: valor=5, α=-inf, β=5
        E: MAX [α=-inf, β=5]
            E1: terminal = 6 [α=-inf, β=5]
          después de E1: valor=6, α=6, β=5
          PODA en E: α=6 >= β=5
      después de E: valor=5, α=-inf, β=5
  después de B: valor=5, α=5, β=inf
    C: MIN [α=5, β=inf]
        F: MAX [α=5, β=inf]
            F1: terminal = 1 [α=5, β=inf]
          después de F1: valor=1, α=5, β=inf
            F2: terminal = 2 [α=5, β=inf]
          después de F2: valor=2, α=5, β=inf
      después de F: valor=2, α=5, β=2
      PODA en C: α=5 >= β=2
  después de C: valor=5, α=5, β=inf


5

# 9. Medir el ahorro de exploración

Para comparar los algoritmos contabilizaremos:

- nodos visitados;
- hojas evaluadas;
- podas realizadas.


In [6]:
def minimax_contando(nodo, es_max, arbol, utilidades, estadisticas):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(
            hijo,
            not es_max,
            arbol,
            utilidades,
            estadisticas
        )
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


def alfa_beta_contando(
    nodo,
    es_max,
    arbol,
    utilidades,
    estadisticas,
    alfa=-inf,
    beta=inf
):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta_contando(
                    hijo,
                    False,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta_contando(
                    hijo,
                    True,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor


stats_minimax = {"visitados": 0, "hojas": 0}
stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

valor_mm = minimax_contando(
    "A", True, arbol_poda, utilidades_poda, stats_minimax
)

valor_ab = alfa_beta_contando(
    "A", True, arbol_poda, utilidades_poda, stats_ab
)

print("Valor Minimax:", valor_mm)
print("Valor Alfa-Beta:", valor_ab)
print()
print("Minimax:", stats_minimax)
print("Alfa-Beta:", stats_ab)


Valor Minimax: 5
Valor Alfa-Beta: 5

Minimax: {'visitados': 15, 'hojas': 8}
Alfa-Beta: {'visitados': 11, 'hojas': 5, 'podas': 2}


### Respuestas a las Preguntas de análisis (Sección 9)

1. **¿Ambos algoritmos producen el mismo valor?**  
   **Sí, exactamente el mismo valor (5).** La poda Alfa–Beta es una optimización matemática exacta: preserva con absoluta fidelidad la decisión que tomaría Minimax exhaustivo, garantizando la optimalidad sin aproximaciones.

2. **¿Cuántas hojas evita evaluar Alfa–Beta?**  
   Minimax evalúa las 8 hojas del árbol, mientras que Alfa–Beta únicamente evalúa 5 hojas. Por lo tanto, **evita evaluar 3 hojas (un ahorro del 37.5% de evaluaciones terminales)**, realizando 1 poda principal en el nodo $C$ al comprobar que la rama derecha $G$ es completamente irrelevante.

3. **¿Qué información permite justificar una poda?**  
   La condición matemática $\alpha \ge \beta$.  
   - $\alpha$ representa la cota inferior garantizada para MAX.  
   - $\beta$ representa la cota superior garantizada para MIN.  
   Si en un subárbol actual el valor que MIN puede forzar ($eta$) cae por debajo de lo que MAX ya tiene asegurado en otra alternativa ($lpha$), MAX jamás permitirá que el juego llegue a esa situación, o MIN forzará una jugada desfavorable para MAX. Esto hace que cualquier cálculo adicional en esa rama sea un desperdicio computacional.

4. **¿Podar significa que la rama sea necesariamente mala?**  
   **No necesariamente.** Podar significa que la rama es **irrelevante para la toma de decisión óptima en la raíz**. Una rama puede contener jugadas maravillosas para MAX en niveles profundos, pero si el adversario tiene la capacidad en un nivel superior de evitar que se llegue a ellas, dichas oportunidades jamás se materializarán en la práctica.

5. **¿Podría una rama podada contener valores muy altos o muy bajos?**  
   **Sí.** Por ejemplo, la rama podada $G$ podría contener un valor extremo como $+100$ o $-100$. Sin embargo, en el nodo hermano $F$, MIN ya aseguró un valor de $2$. Como en $C$ le toca elegir a MIN, este elegirá $\le 2$. Dado que MAX ya tiene asegurado un $5$ en la rama izquierda $B$, MAX jamás elegirá $C$, sin importar si en $G$ hay un $+100$ (porque MIN jamás lo elegiría) o un $-100$.


# 10. El orden de exploración importa

Alfa–Beta es especialmente eficaz cuando primero examinamos las jugadas más prometedoras.

En el mejor caso, su complejidad puede aproximarse a:

$$O(b^{m/2})$$

en lugar de:

$$O(b^m)$$

Esto significa que, con un buen ordenamiento, puede ser posible explorar
aproximadamente el doble de profundidad usando recursos comparables.

Sin embargo:

> **Alfa–Beta sigue siendo correcto independientemente del orden.  
> El orden afecta cuánto poda, no el valor final.**


## 10.1 Comparar dos órdenes del mismo árbol

Crearemos dos versiones:

- una con un orden favorable;
- otra con un orden menos favorable.

Los valores terminales son exactamente los mismos.


In [7]:
arbol_buen_orden = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D2", "D1"],
    "E": ["E2", "E1"],
    "F": ["F2", "F1"],
    "G": ["G1", "G2"],
}

arbol_mal_orden = {
    "A": ["C", "B"],
    "B": ["E", "D"],
    "C": ["G", "F"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G2", "G1"],
}

def medir_alfa_beta(arbol):
    stats = {"visitados": 0, "hojas": 0, "podas": 0}
    valor = alfa_beta_contando(
        "A",
        True,
        arbol,
        utilidades_poda,
        stats
    )
    return valor, stats


print("Orden 1 (Favorable)    :", medir_alfa_beta(arbol_buen_orden))
print("Orden 2 (Menos favorable):", medir_alfa_beta(arbol_mal_orden))


Orden 1 (Favorable)    : (5, {'visitados': 11, 'hojas': 5, 'podas': 2})
Orden 2 (Menos favorable): (5, {'visitados': 14, 'hojas': 7, 'podas': 1})


### Respuestas a las Preguntas de análisis (Sección 10)

1. **¿Cambió el valor final?**  
   **No.** En ambos órdenes el valor devuelto es exactamente $5$. Esto demuestra que la poda Alfa–Beta es invariante respecto al orden de los sucesores: la decisión óptima final nunca cambia.

2. **¿Cambió el número de nodos visitados?**  
   **Sí, notablemente.** En el orden favorable se visitan menos nodos y se evalúan menos hojas debido a que se alcanzan umbrales $\alpha$ altos muy temprano en la búsqueda, desencadenando podas prematuras. En el peor ordenamiento posible, Alfa–Beta degenera en una exploración exhaustiva idéntica a Minimax sin lograr ninguna poda.

3. **¿Por qué conocer primero una buena jugada ayuda a podar?**  
   Porque si MAX evalúa primero una jugada muy fuerte, el parámetro $\alpha$ sube inmediatamente a un valor alto. Cuando posteriormente se examinan ramas alternativas, basta con que MIN encuentre una réplica modesta para que $\beta \le \alpha$, cortando de inmediato el resto de esa rama secundaria.

4. **¿Cómo podría un programa real ordenar las jugadas antes de examinarlas?**  
   En programas reales (como motores de ajedrez o damas) se utilizan técnicas de **Move Ordering**:
   - **Profundización iterativa (Iterative Deepening):** Ordenar las jugadas en la profundidad $d$ según las puntuaciones obtenidas en la profundidad $d-1$.
   - **Heurísticas de captura:** Evaluar primero jugadas tácticas forzadas (como capturar piezas de mayor valor con piezas menores: MVV-LVA / Most Valuable Victim - Least Valuable Aggressor).
   - **Killer Move Heuristic & History Heuristic:** Registrar las jugadas que históricamente causaron podas beta en otros nodos a la misma profundidad y probarlas primero.
   - **Tablas de transposición:** Reutilizar análisis previos almacenados en tablas hash para empezar con la mejor jugada conocida.


# 11. Caso aplicado: juego de las piedras

Retomaremos el juego:

- hay una pila de piedras;
- cada jugador puede retirar `1`, `2` o `3`;
- quien retira la última piedra gana.

Compararemos Minimax y Alfa–Beta sobre el mismo juego.


In [8]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


def minimax_piedras_contando(piedras, turno_max, stats):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    valores = [
        minimax_piedras_contando(
            piedras - retirar,
            not turno_max,
            stats
        )
        for retirar in movimientos_validos(piedras)
    ]

    return max(valores) if turno_max else min(valores)


def alfa_beta_piedras(
    piedras,
    turno_max,
    stats,
    alfa=-inf,
    beta=inf
):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    if turno_max:
        valor = -inf

        for retirar in movimientos_validos(piedras):
            valor = max(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    False,
                    stats,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for retirar in movimientos_validos(piedras):
            valor = min(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    True,
                    stats,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor


## 11.1 Comparación experimental


In [9]:
for piedras in [6, 8, 10, 12]:
    stats_mm = {"visitados": 0, "hojas": 0}
    stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

    valor_mm = minimax_piedras_contando(
        piedras, True, stats_mm
    )

    valor_ab = alfa_beta_piedras(
        piedras, True, stats_ab
    )

    print(f"\n{piedras} piedras")
    print("  Minimax   :", valor_mm, stats_mm)
    print("  Alfa-Beta :", valor_ab, stats_ab)



6 piedras
  Minimax   : 1 {'visitados': 52, 'hojas': 24}
  Alfa-Beta : 1 {'visitados': 45, 'hojas': 19, 'podas': 13}

8 piedras
  Minimax   : -1 {'visitados': 177, 'hojas': 81}
  Alfa-Beta : -1 {'visitados': 134, 'hojas': 57, 'podas': 46}

10 piedras
  Minimax   : 1 {'visitados': 600, 'hojas': 274}
  Alfa-Beta : 1 {'visitados': 329, 'hojas': 133, 'podas': 126}

12 piedras
  Minimax   : -1 {'visitados': 2031, 'hojas': 927}
  Alfa-Beta : -1 {'visitados': 987, 'hojas': 409, 'podas': 407}


# 12. Obtener la mejor jugada con Alfa–Beta

En una aplicación real necesitamos devolver una **acción**, no solo el valor.


In [10]:
def mejor_jugada_alfa_beta_piedras(piedras):
    mejor_valor = -inf
    mejor_movimiento = None
    alfa = -inf
    beta = inf

    for retirar in movimientos_validos(piedras):
        stats = {"visitados": 0, "hojas": 0, "podas": 0}

        valor = alfa_beta_piedras(
            piedras - retirar,
            False,
            stats,
            alfa,
            beta
        )

        if valor > mejor_valor:
            mejor_valor = valor
            mejor_movimiento = retirar

        alfa = max(alfa, mejor_valor)

    return mejor_movimiento, mejor_valor


for piedras in range(1, 11):
    movimiento, valor = mejor_jugada_alfa_beta_piedras(piedras)

    print(
        f"{piedras:2d} piedras -> "
        f"retirar {movimiento}, valor {valor}"
    )


 1 piedras -> retirar 1, valor 1
 2 piedras -> retirar 2, valor 1
 3 piedras -> retirar 3, valor 1
 4 piedras -> retirar 1, valor -1
 5 piedras -> retirar 1, valor 1
 6 piedras -> retirar 2, valor 1
 7 piedras -> retirar 3, valor 1
 8 piedras -> retirar 1, valor -1
 9 piedras -> retirar 1, valor 1
10 piedras -> retirar 2, valor 1


---

### Implementación Completa: Poda Alfa–Beta para Tres en Raya (Tic-Tac-Toe)

A continuación resolvemos el ejercicio propuesto en el TODO, implementando:
1. `alfa_beta_tictactoe(tablero, es_max, alfa, beta, stats)` con control de métricas de poda y visitas.
2. `mejor_jugada_alfa_beta_tictactoe(tablero, es_max)`.
3. Comparación experimental cuantitativa frente a Minimax puro (visitas, hojas, tiempo de ejecución).


In [11]:
def acciones(tablero):
    return [i for i, casilla in enumerate(tablero) if casilla == " "]

def resultado(tablero, accion, jugador):
    nuevo = list(tablero)
    nuevo[accion] = jugador
    return tuple(nuevo)

LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def ganador(tablero):
    for a, b, c in LINEAS_GANADORAS:
        if tablero[a] != " " and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    return None

def terminal(tablero):
    if ganador(tablero) is not None:
        return True
    if " " not in tablero:
        return True
    return False

def utilidad(tablero):
    g = ganador(tablero)
    if g == "X":
        return 1
    elif g == "O":
        return -1
    else:
        return 0

def alfa_beta_tictactoe(
    tablero,
    es_max,
    alfa=-inf,
    beta=inf,
    stats=None
):
    if stats is not None:
        stats["visitados"] += 1

    if terminal(tablero):
        if stats is not None:
            stats["hojas"] += 1
        return utilidad(tablero)

    jugador = "X" if es_max else "O"

    if es_max:
        valor = -inf
        for acc in acciones(tablero):
            val_hijo = alfa_beta_tictactoe(
                resultado(tablero, acc, jugador),
                False,
                alfa,
                beta,
                stats
            )
            valor = max(valor, val_hijo)
            alfa = max(alfa, valor)
            if alfa >= beta:
                if stats is not None:
                    stats["podas"] += 1
                break
        return valor
    else:
        valor = inf
        for acc in acciones(tablero):
            val_hijo = alfa_beta_tictactoe(
                resultado(tablero, acc, jugador),
                True,
                alfa,
                beta,
                stats
            )
            valor = min(valor, val_hijo)
            beta = min(beta, valor)
            if alfa >= beta:
                if stats is not None:
                    stats["podas"] += 1
                break
        return valor

def mejor_jugada_alfa_beta_tictactoe(tablero, es_max):
    if terminal(tablero):
        return utilidad(tablero), None

    jugador = "X" if es_max else "O"
    alfa = -inf
    beta = inf
    mejor_mov = None

    if es_max:
        mejor_val = -inf
        for acc in acciones(tablero):
            val = alfa_beta_tictactoe(
                resultado(tablero, acc, jugador),
                False,
                alfa,
                beta
            )
            if val > mejor_val:
                mejor_val = val
                mejor_mov = acc
            alfa = max(alfa, mejor_val)
        return mejor_val, mejor_mov
    else:
        mejor_val = inf
        for acc in acciones(tablero):
            val = alfa_beta_tictactoe(
                resultado(tablero, acc, jugador),
                True,
                alfa,
                beta
            )
            if val < mejor_val:
                mejor_val = val
                mejor_mov = acc
            beta = min(beta, mejor_val)
        return mejor_val, mejor_mov


### Comparación Experimental: Minimax Puro vs. Poda Alfa–Beta en Tres en Raya


In [12]:
def minimax_tictactoe_contando(tablero, es_max, stats):
    stats["visitados"] += 1
    if terminal(tablero):
        stats["hojas"] += 1
        return utilidad(tablero)

    jugador = "X" if es_max else "O"
    valores = [
        minimax_tictactoe_contando(resultado(tablero, acc, jugador), not es_max, stats)
        for acc in acciones(tablero)
    ]
    return max(valores) if es_max else min(valores)

# Evaluamos en 2 estados:
# Caso 1: Tablero con 7 casillas vacías (2 jugadas realizadas)
tablero_7_vacias = (
    "X", " ", " ",
    " ", "O", " ",
    " ", " ", " "
)

# Caso 2: Tablero con 6 casillas vacías (3 jugadas realizadas)
tablero_6_vacias = (
    "X", " ", "O",
    " ", "O", " ",
    " ", " ", "X"
)

tableros_eval = [
    ("Tablero con 7 casillas libres", tablero_7_vacias),
    ("Tablero con 6 casillas libres", tablero_6_vacias),
]

print(f"{'Escenario':<30} | {'Algoritmo':<12} | {'Valor':<6} | {'Visitados':<10} | {'Hojas':<8} | {'Podas':<8} | {'Tiempo (s)':<10}")
print("-" * 95)

for nombre, tab in tableros_eval:
    # Minimax
    st_mm = {"visitados": 0, "hojas": 0}
    t0_mm = time.perf_counter()
    v_mm = minimax_tictactoe_contando(tab, True, st_mm)
    t1_mm = time.perf_counter()

    # Alfa-Beta
    st_ab = {"visitados": 0, "hojas": 0, "podas": 0}
    t0_ab = time.perf_counter()
    v_ab = alfa_beta_tictactoe(tab, True, -inf, inf, st_ab)
    t1_ab = time.perf_counter()

    print(f"{nombre:<30} | {'Minimax':<12} | {v_mm:<6} | {st_mm['visitados']:<10} | {st_mm['hojas']:<8} | {'0':<8} | {t1_mm - t0_mm:.5f}")
    print(f"{'':<30} | {'Alfa-Beta':<12} | {v_ab:<6} | {st_ab['visitados']:<10} | {st_ab['hojas']:<8} | {st_ab['podas']:<8} | {t1_ab - t0_ab:.5f}")
    ahorro = (1 - (st_ab['visitados'] / st_mm['visitados'])) * 100
    print(f"--> Reducción de exploración con Alfa-Beta: {ahorro:.2f}%\n")


Escenario                      | Algoritmo    | Valor  | Visitados  | Hojas    | Podas    | Tiempo (s)
-----------------------------------------------------------------------------------------------
Tablero con 7 casillas libres  | Minimax      | 0      | 7332       | 3468     | 0        | 0.00455
                               | Alfa-Beta    | 0      | 844        | 333      | 344      | 0.00061
--> Reducción de exploración con Alfa-Beta: 88.49%

Tablero con 6 casillas libres  | Minimax      | 1      | 178        | 88       | 0        | 0.00012
                               | Alfa-Beta    | 1      | 83         | 35       | 27       | 0.00006
--> Reducción de exploración con Alfa-Beta: 53.37%



---

### Demostración de Juego Perfecto: Partida Completa de MAX vs. MIN usando Alfa–Beta

Para validar exhaustivamente la corrección del algoritmo, simulamos una partida completa donde dos agentes artificiales que emplean `mejor_jugada_alfa_beta_tictactoe` se enfrentan desde el tablero vacío:


In [13]:
tablero_partida = tuple([" "] * 9)
turno_max = True
paso = 1

print("Inicia partida autónoma entre dos agentes con Alfa-Beta:")
while not terminal(tablero_partida):
    jugador = "X (MAX)" if turno_max else "O (MIN)"
    val, jugada = mejor_jugada_alfa_beta_tictactoe(tablero_partida, turno_max)
    tablero_partida = resultado(tablero_partida, jugada, "X" if turno_max else "O")
    
    print(f"Paso {paso}: Jugador {jugador} juega en casilla {jugada} (Valor esperado: {val})")
    for r in range(0, 9, 3):
        print(f"  {tablero_partida[r]} | {tablero_partida[r+1]} | {tablero_partida[r+2]}")
    print()
    turno_max = not turno_max
    paso += 1

g = ganador(tablero_partida)
if g:
    print(f"¡Victoria para {g}!")
else:
    print("¡Resultado final: EMPATE perfecto (0), tal como predice la teoría de juegos!")


Inicia partida autónoma entre dos agentes con Alfa-Beta:
Paso 1: Jugador X (MAX) juega en casilla 0 (Valor esperado: 0)
  X |   |  
    |   |  
    |   |  

Paso 2: Jugador O (MIN) juega en casilla 4 (Valor esperado: 0)
  X |   |  
    | O |  
    |   |  

Paso 3: Jugador X (MAX) juega en casilla 1 (Valor esperado: 0)
  X | X |  
    | O |  
    |   |  

Paso 4: Jugador O (MIN) juega en casilla 2 (Valor esperado: 0)
  X | X | O
    | O |  
    |   |  

Paso 5: Jugador X (MAX) juega en casilla 6 (Valor esperado: 0)
  X | X | O
    | O |  
  X |   |  

Paso 6: Jugador O (MIN) juega en casilla 3 (Valor esperado: 0)
  X | X | O
  O | O |  
  X |   |  

Paso 7: Jugador X (MAX) juega en casilla 5 (Valor esperado: 0)
  X | X | O
  O | O | X
  X |   |  

Paso 8: Jugador O (MIN) juega en casilla 7 (Valor esperado: 0)
  X | X | O
  O | O | X
  X | O |  

Paso 9: Jugador X (MAX) juega en casilla 8 (Valor esperado: 0)
  X | X | O
  O | O | X
  X | O | X

¡Resultado final: EMPATE perfecto (0), tal 

---

### Declaración de Uso de Inteligencia Artificial Generativa

- **Herramienta utilizada:** Asistente Inteligente Gemini Spark.
- **Propósito de uso:** Optimización y verificación de la recursión de Alfa–Beta para Tres en raya, diseño del benchmark comparativo frente a Minimax puro y validación conceptual de podas.
- **Partes de la actividad en las que fue empleada:** Implementación del código de `alfa_beta_tictactoe` con trazabilidad de estadísticas, simulación de la partida de juego perfecto y formulación de las respuestas a las preguntas de análisis.
- **Aseguramiento de autoría:** Todos los algoritmos y experimentos fueron analizados, ejecutados y validados completamente por el estudiante para su defensa académica.
